# Comprehensive Data Exploration: Alzheimer's Disease Detection

This notebook provides an in-depth analysis of the MRI biomarker dataset for early Alzheimer's disease detection.

## Objectives
- Understand dataset structure and quality
- Analyze feature distributions and relationships
- Identify patterns and correlations
- Prepare data for machine learning modeling


## 1. Environment Setup and Data Loading


In [ ]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


In [ ]:
# Load the dataset
data_path = Path('../data/oasis_longitudinal.csv')
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {list(df.columns)}")
df.head()


## 2. Initial Data Assessment


In [ ]:
# Comprehensive data info
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nTotal records: {len(df):,}")
print(f"Total features: {len(df.columns)}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Data types
print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)
print(df.dtypes)


In [ ]:
# Missing values analysis
missing_data = df.isnull().sum()
missing_pct = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': missing_data.index,
    'Missing Count': missing_data.values,
    'Percentage': missing_pct.values
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    print("Missing Values Summary:")
    print(missing_df.to_string(index=False))
    
    # Visualize missing data
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=missing_df, x='Column', y='Percentage', ax=ax)
    ax.set_title('Missing Data Percentage by Column', fontsize=14, fontweight='bold')
    ax.set_xlabel('Column Name', fontsize=12)
    ax.set_ylabel('Missing Percentage (%)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("✓ No missing values found in the dataset")


## 3. Data Preprocessing and Feature Engineering


In [ ]:
# Prepare data for analysis
df_analysis = df.copy()

# Filter to first visit only for consistency
if 'Visit' in df_analysis.columns:
    df_analysis = df_analysis[df_analysis['Visit'] == 1].reset_index(drop=True)
    print(f"Records after filtering to first visit: {len(df_analysis)}")

# Encode categorical variables
if 'M/F' in df_analysis.columns:
    df_analysis['Gender_Encoded'] = df_analysis['M/F'].map({'F': 0, 'M': 1})

# Process target variable
if 'Group' in df_analysis.columns:
    df_analysis['Group'] = df_analysis['Group'].replace(['Converted'], 'Demented')
    df_analysis['Dementia_Status'] = df_analysis['Group'].map({'Demented': 1, 'Nondemented': 0})

# Handle missing SES values with median imputation by education level
if 'SES' in df_analysis.columns and df_analysis['SES'].isnull().sum() > 0:
    df_analysis['SES'] = df_analysis['SES'].fillna(
        df_analysis.groupby('EDUC')['SES'].transform('median')
    )
    print(f"✓ Imputed {df_analysis['SES'].isnull().sum()} missing SES values")

# Remove unnecessary columns
cols_to_drop = ['MRI ID', 'Visit', 'Hand']
existing_cols = [col for col in cols_to_drop if col in df_analysis.columns]
if existing_cols:
    df_analysis = df_analysis.drop(existing_cols, axis=1)

print(f"\nFinal dataset shape: {df_analysis.shape}")


## 4. Target Variable Distribution


In [ ]:
# Analyze target variable distribution
if 'Dementia_Status' in df_analysis.columns:
    target_counts = df_analysis['Dementia_Status'].value_counts()
    target_pct = df_analysis['Dementia_Status'].value_counts(normalize=True) * 100
    
    print("Target Variable Distribution:")
    print("-" * 40)
    for status, count in target_counts.items():
        status_name = 'Demented' if status == 1 else 'Nondemented'
        print(f"{status_name}: {count} ({target_pct[status]:.1f}%)")
    
    # Visualize distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Count plot
    sns.countplot(data=df_analysis, x='Dementia_Status', ax=axes[0], palette='Set2')
    axes[0].set_title('Dementia Status Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Status (0=Nondemented, 1=Demented)', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    
    # Pie chart
    axes[1].pie(target_counts.values, labels=['Nondemented', 'Demented'], 
                autopct='%1.1f%%', startangle=90, colors=['#66b3ff', '#ff9999'])
    axes[1].set_title('Class Proportions', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()


## 5. Numerical Features Analysis


In [ ]:
# Select numerical features for analysis
numerical_features = ['Age', 'EDUC', 'SES', 'MMSE', 'eTIV', 'nWBV', 'ASF']
available_features = [f for f in numerical_features if f in df_analysis.columns]

# Statistical summary
print("Statistical Summary of Numerical Features:")
print("=" * 80)
stats_summary = df_analysis[available_features].describe()
print(stats_summary.round(2))


In [ ]:
# Distribution plots for numerical features
n_features = len(available_features)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5*n_rows))
axes = axes.flatten() if n_features > 1 else [axes]

for idx, feature in enumerate(available_features):
    ax = axes[idx]
    
    # Create histogram with KDE
    sns.histplot(data=df_analysis, x=feature, kde=True, ax=ax, bins=30)
    ax.set_title(f'{feature} Distribution', fontsize=12, fontweight='bold')
    ax.set_xlabel(feature, fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(n_features, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()


## 6. Feature Relationships with Target Variable


In [ ]:
# Compare feature distributions by dementia status
if 'Dementia_Status' in df_analysis.columns:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for idx, feature in enumerate(available_features[:8]):
        ax = axes[idx]
        
        # Box plot comparison
        sns.boxplot(data=df_analysis, x='Dementia_Status', y=feature, ax=ax, palette='Set2')
        ax.set_title(f'{feature} by Dementia Status', fontsize=11, fontweight='bold')
        ax.set_xlabel('Status (0=Nondemented, 1=Demented)', fontsize=9)
        ax.set_ylabel(feature, fontsize=9)
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for idx in range(len(available_features), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()


## 7. Correlation Analysis


In [ ]:
# Calculate correlation matrix
features_for_corr = available_features + ['Dementia_Status']
correlation_matrix = df_analysis[features_for_corr].corr()

# Visualize correlation heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='coolwarm', center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Show correlations with target
target_correlations = correlation_matrix['Dementia_Status'].sort_values(ascending=False)
target_correlations = target_correlations[target_correlations.index != 'Dementia_Status']

print("\nCorrelation with Dementia Status:")
print("-" * 50)
for feature, corr in target_correlations.items():
    print(f"{feature:15s}: {corr:6.3f}")


## 8. Demographic Analysis


In [ ]:
# Gender analysis
if 'Gender_Encoded' in df_analysis.columns and 'Dementia_Status' in df_analysis.columns:
    gender_dementia = pd.crosstab(df_analysis['Gender_Encoded'], df_analysis['Dementia_Status'], 
                                  margins=True, normalize='index') * 100
    
    print("Dementia Prevalence by Gender:")
    print("-" * 50)
    print(gender_dementia.round(2))
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Count by gender and status
    gender_status_counts = pd.crosstab(df_analysis['Gender_Encoded'], df_analysis['Dementia_Status'])
    gender_status_counts.plot(kind='bar', stacked=True, ax=axes[0], color=['#66b3ff', '#ff9999'])
    axes[0].set_title('Dementia Cases by Gender', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Gender (0=Female, 1=Male)', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].legend(['Nondemented', 'Demented'])
    axes[0].tick_params(axis='x', rotation=0)
    
    # Age distribution by status
    if 'Age' in df_analysis.columns:
        sns.violinplot(data=df_analysis, x='Dementia_Status', y='Age', ax=axes[1], palette='Set2')
        axes[1].set_title('Age Distribution by Dementia Status', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Status (0=Nondemented, 1=Demented)', fontsize=12)
        axes[1].set_ylabel('Age (years)', fontsize=12)
    
    plt.tight_layout()
    plt.show()


## 9. Key Insights and Summary


In [ ]:
print("=" * 60)
print("KEY INSIGHTS FROM DATA EXPLORATION")
print("=" * 60)

insights = [
    f"Total samples analyzed: {len(df_analysis)}",
    f"Features available: {len(available_features)}",
    f"Missing values handled: {df_analysis.isnull().sum().sum()}",
]

if 'Dementia_Status' in df_analysis.columns:
    dementia_rate = df_analysis['Dementia_Status'].mean() * 100
    insights.append(f"Dementia prevalence: {dementia_rate:.1f}%")

if 'MMSE' in df_analysis.columns:
    mmse_diff = df_analysis[df_analysis['Dementia_Status']==0]['MMSE'].mean() - \
                df_analysis[df_analysis['Dementia_Status']==1]['MMSE'].mean()
    insights.append(f"MMSE score difference (Nondemented - Demented): {mmse_diff:.2f}")

for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

print("\n" + "=" * 60)
print("Data is ready for machine learning modeling!")
print("=" * 60)
